In [1]:
import os
import sys

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, coalesce, count, countDistinct, when, lit,
    sum as spark_sum, avg, min as spark_min, max as spark_max,
    to_date, datediff, current_date, size, explode
)

In [24]:
RAW_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

# Ambiente spark

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("offer_personalization_data_processing") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/14 23:28:43 WARN Utils: Your hostname, jorel.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.3 instead (on interface en0)
26/05/14 23:28:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 23:28:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.0.2'

In [ ]:

offers_path = f"{RAW_PATH}/offers.json"
customers_path = f"{RAW_PATH}/profile.json"
transactions_path = f"{RAW_PATH}/transactions.json"

offers = spark.read.option("multiline", "true").json(offers_path)
customers = spark.read.option("multiline", "true").json(customers_path)
transactions = spark.read.option("multiline", "true").json(transactions_path)

In [5]:
print("Offers:", offers.count())
print("Customers:", customers.count())
print("Transactions:", transactions.count())

offers.printSchema()
customers.printSchema()
transactions.printSchema()

Offers: 10
Customers: 17000
Transactions: 306534
root
 |-- channels: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- discount_value: long (nullable = true)
 |-- duration: double (nullable = true)
 |-- id: string (nullable = true)
 |-- min_value: long (nullable = true)
 |-- offer_type: string (nullable = true)

root
 |-- age: long (nullable = true)
 |-- credit_card_limit: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- id: string (nullable = true)
 |-- registered_on: string (nullable = true)

root
 |-- account_id: string (nullable = true)
 |-- event: string (nullable = true)
 |-- time_since_test_start: double (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- offer id: string (nullable = true)
 |    |-- offer_id: string (nullable = true)
 |    |-- reward: double (nullable = true)



# Processamento nos dados

## 1. Processamento inicial nos dados de clientes

In [6]:
customers_clean = (
    customers
    .withColumn("registered_on", to_date(col("registered_on"), "yyyyMMdd"))
    .withColumn("account_age_days", datediff(current_date(), col("registered_on")))
    .withColumn(
        "gender",
        when(col("gender").isNull(), "unknown")
        .otherwise(col("gender"))
    )
    .withColumn(
        "age",
        when((col("age").isNull()) | (col("age") >= 100), None)
        .otherwise(col("age"))
    )
)

In [7]:
customers_clean.printSchema()
customers_clean.show(5, truncate=False)

root
 |-- age: long (nullable = true)
 |-- credit_card_limit: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- id: string (nullable = true)
 |-- registered_on: date (nullable = true)
 |-- account_age_days: integer (nullable = true)

+----+-----------------+-------+--------------------------------+-------------+----------------+
|age |credit_card_limit|gender |id                              |registered_on|account_age_days|
+----+-----------------+-------+--------------------------------+-------------+----------------+
|NULL|NULL             |unknown|68be06ca386d4c31939f3a4f0e3dd783|2017-02-12   |3378            |
|55  |112000.0         |F      |0610b486422d4921ae7d2bf64640c50b|2017-07-15   |3225            |
|NULL|NULL             |unknown|38fe809add3b4fcf9315a9694bb96ff5|2018-07-12   |2863            |
|75  |100000.0         |F      |78afa995795e4d85b5d9ceeca43f5fef|2017-05-09   |3292            |
|NULL|NULL             |unknown|a03223e636434f42ac4c3df47e8bac43|2017

## 2. Processamento inicial nos dados de campanha

In [8]:
offers_clean = (
    offers
    .withColumn("duration_days", col("duration").cast("int"))
    .withColumn("discount_value", col("discount_value").cast("double"))
    .withColumn("min_value", col("min_value").cast("double"))
    .withColumn("num_channels", size(col("channels")))
    .withColumn("is_bogo", when(col("offer_type") == "bogo", 1).otherwise(0))
    .withColumn("is_discount", when(col("offer_type") == "discount", 1).otherwise(0))
    .withColumn("is_informational", when(col("offer_type") == "informational", 1).otherwise(0))
)

In [9]:
offers_channels = (
    offers_clean
    .select("id", "offer_type", explode(col("channels")).alias("channel"))
)

In [10]:
offers_clean.show(5, truncate=False)
offers_channels.show(10, truncate=False)

+----------------------------+--------------+--------+--------------------------------+---------+-------------+-------------+------------+-------+-----------+----------------+
|channels                    |discount_value|duration|id                              |min_value|offer_type   |duration_days|num_channels|is_bogo|is_discount|is_informational|
+----------------------------+--------------+--------+--------------------------------+---------+-------------+-------------+------------+-------+-----------+----------------+
|[email, mobile, social]     |10.0          |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10.0     |bogo         |7            |3           |1      |0          |0               |
|[web, email, mobile, social]|10.0          |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10.0     |bogo         |5            |4           |1      |0          |0               |
|[web, email, mobile]        |0.0           |4.0     |3f207df678b143eea3cee63160fa8bed|0.0      |informational|4        

## 3. Processamento inicial nos dados de transação

In [11]:
transactions_clean = (
    transactions
    .withColumn(
        "offer_id",
        coalesce(
            col("value.offer_id"),
            col("value.`offer id`")
        )
    )
    .withColumn("amount", col("value.amount"))
    .withColumn("reward", col("value.reward"))
    .withColumn("time_since_test_start", col("time_since_test_start").cast("int"))
    .drop("value")
)

In [12]:
transactions_clean.groupBy("event").count().show(truncate=False)
transactions_clean.show(5, truncate=False)

+---------------+------+
|event          |count |
+---------------+------+
|transaction    |138953|
|offer received |76277 |
|offer completed|33579 |
|offer viewed   |57725 |
+---------------+------+

+--------------------------------+--------------+---------------------+--------------------------------+------+------+
|account_id                      |event         |time_since_test_start|offer_id                        |amount|reward|
+--------------------------------+--------------+---------------------+--------------------------------+------+------+
|78afa995795e4d85b5d9ceeca43f5fef|offer received|0                    |9b98b8c7a33c4b65b9aebfe6a799e6d9|NULL  |NULL  |
|a03223e636434f42ac4c3df47e8bac43|offer received|0                    |0b1e1539f2cc45b7b9fa7c272da2e1d7|NULL  |NULL  |
|e2127556f4f64592b11af22de27a7932|offer received|0                    |2906b810c7d4411798c6938adc9daaa5|NULL  |NULL  |
|8ec6ce2a7e7949b1bf142def7d0e0586|offer received|0                    |fafdcd668e3743

### 3.1 Separação dos eventos

In [13]:
transactions_only = transactions_clean.filter(col("event") == "transaction")

offers_received = transactions_clean.filter(col("event") == "offer received")

offers_viewed = transactions_clean.filter(col("event") == "offer viewed")

offers_completed = transactions_clean.filter(col("event") == "offer completed")

### 3.2 Métricas de compra por cliente

In [14]:
max_test_day = transactions_clean.agg(
    spark_max("time_since_test_start").alias("max_day")
).collect()[0]["max_day"]

customer_transaction_metrics = (
    transactions_only
    .groupBy("account_id")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("amount").alias("total_spent"),
        avg("amount").alias("avg_ticket"),
        spark_min("amount").alias("min_ticket"),
        spark_max("amount").alias("max_ticket"),
        spark_min("time_since_test_start").alias("first_transaction_day"),
        spark_max("time_since_test_start").alias("last_transaction_day")
    )
    .withColumn(
        "recency_days",
        lit(max_test_day) - col("last_transaction_day")
    )
)

### 3.3 Métricas de oferta por cliente

In [15]:
customer_offer_metrics = (
    transactions_clean
    .filter(col("event").isin("offer received", "offer viewed", "offer completed"))
    .groupBy("account_id")
    .agg(
        count(when(col("event") == "offer received", True)).alias("offers_received"),
        count(when(col("event") == "offer viewed", True)).alias("offers_viewed"),
        count(when(col("event") == "offer completed", True)).alias("offers_completed"),
        countDistinct("offer_id").alias("distinct_offers")
    )
    .withColumn(
        "view_rate",
        when(col("offers_received") > 0, col("offers_viewed") / col("offers_received"))
        .otherwise(0)
    )
    .withColumn(
        "completion_rate",
        when(col("offers_received") > 0, col("offers_completed") / col("offers_received"))
        .otherwise(0)
    )
)

### 3.4 Base cliente consolidada

In [16]:
customer_features = (
    customers_clean.alias("c")
    .join(
        customer_transaction_metrics.alias("t"),
        col("c.id") == col("t.account_id"),
        "left"
    )
    .join(
        customer_offer_metrics.alias("o"),
        col("c.id") == col("o.account_id"),
        "left"
    )
    .drop(col("t.account_id"))
    .drop(col("o.account_id"))
)

In [17]:
customer_features = (
    customer_features
    .fillna({
        "total_transactions": 0,
        "total_spent": 0,
        "avg_ticket": 0,
        "min_ticket": 0,
        "max_ticket": 0,
        "offers_received": 0,
        "offers_viewed": 0,
        "offers_completed": 0,
        "distinct_offers": 0,
        "view_rate": 0,
        "completion_rate": 0
    })
)

### 3.5 Base cliente oferta

In [18]:
offer_interactions = (
    offers_received
    .select(
        col("account_id"),
        col("offer_id"),
        col("time_since_test_start").alias("offer_received_day")
    )
    .join(
        offers_viewed
        .select(
            col("account_id").alias("viewed_account_id"),
            col("offer_id").alias("viewed_offer_id"),
            col("time_since_test_start").alias("offer_viewed_day")
        ),
        (col("account_id") == col("viewed_account_id")) &
        (col("offer_id") == col("viewed_offer_id")),
        "left"
    )
    .join(
        offers_completed
        .select(
            col("account_id").alias("completed_account_id"),
            col("offer_id").alias("completed_offer_id"),
            col("time_since_test_start").alias("offer_completed_day"),
            col("reward").alias("reward_received")
        ),
        (col("account_id") == col("completed_account_id")) &
        (col("offer_id") == col("completed_offer_id")),
        "left"
    )
    .drop("viewed_account_id", "viewed_offer_id", "completed_account_id", "completed_offer_id")
    .withColumn(
        "was_viewed",
        when(col("offer_viewed_day").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "was_completed",
        when(col("offer_completed_day").isNotNull(), 1).otherwise(0)
    )
)

In [19]:
customer_offer_dataset = (
    offer_interactions.alias("i")
    .join(
        customer_features.alias("c"),
        col("i.account_id") == col("c.id"),
        "left"
    )
    .join(
        offers_clean.alias("o"),
        col("i.offer_id") == col("o.id"),
        "left"
    )
    .drop(col("c.id"))
    .drop(col("o.id"))
)

In [20]:
customer_offer_dataset.printSchema()
customer_offer_dataset.show(5, truncate=False)

root
 |-- account_id: string (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- offer_received_day: integer (nullable = true)
 |-- offer_viewed_day: integer (nullable = true)
 |-- offer_completed_day: integer (nullable = true)
 |-- reward_received: double (nullable = true)
 |-- was_viewed: integer (nullable = false)
 |-- was_completed: integer (nullable = false)
 |-- age: long (nullable = true)
 |-- credit_card_limit: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- registered_on: date (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- avg_ticket: double (nullable = true)
 |-- min_ticket: double (nullable = true)
 |-- max_ticket: double (nullable = true)
 |-- first_transaction_day: integer (nullable = true)
 |-- last_transaction_day: integer (nullable = true)
 |-- recency_days: integer (nullable = true)
 |-- offers_received: long (nullable = tru

+--------------------------------+--------------------------------+------------------+----------------+-------------------+---------------+----------+-------------+----+-----------------+-------+-------------+----------------+------------------+-----------+------------------+----------+----------+---------------------+--------------------+------------+---------------+-------------+----------------+---------------+---------+------------------+----------------------------+--------------+--------+---------+----------+-------------+------------+-------+-----------+----------------+
|account_id                      |offer_id                        |offer_received_day|offer_viewed_day|offer_completed_day|reward_received|was_viewed|was_completed|age |credit_card_limit|gender |registered_on|account_age_days|total_transactions|total_spent|avg_ticket        |min_ticket|max_ticket|first_transaction_day|last_transaction_day|recency_days|offers_received|offers_viewed|offers_completed|distinct_offer

## Validação final

In [21]:
print("customers_clean:", customers_clean.count())
print("offers_clean:", offers_clean.count())
print("transactions_clean:", transactions_clean.count())
print("customer_features:", customer_features.count())
print("customer_offer_dataset:", customer_offer_dataset.count())

customers_clean: 17000
offers_clean: 10
transactions_clean: 306534
customer_features: 17000


customer_offer_dataset: 115609


In [22]:
customer_offer_dataset.groupBy("was_completed").count().show()

+-------------+-----+
|was_completed|count|
+-------------+-----+
|            1|67397|
|            0|48212|
+-------------+-----+



## Salvando parquets

In [25]:
customers_clean.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customers_clean.parquet"
)

offers_clean.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/offers_clean.parquet"
)

offers_channels.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/offers_channels.parquet"
)

transactions_clean.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/transactions_clean.parquet"
)

customer_features.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customer_features.parquet"
)

customer_offer_dataset.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customer_offer_dataset.parquet"
)

## Resumo

### Customers
- Tratamento e padronização de atributos cadastrais dos clientes
- Parsing de datas
- Tratamento de valores ausentes

### Offers
- Padronização e enriquecimento das informações das ofertas
- Ajuste de tipos
- Criação de indicadores por tipo de campanha e expansão dos canais de comunicação para análise posterior

### Transactions
- Normalização da estrutura de eventos transacionais
- Padronização de identificadores de ofertas
- Extração de atributos relevantes
- Criação de métricas comportamentais relacionadas a compras e engajamento com campanhas.

### Analytical Dataset
- Construção da base analítica consolidada unificando clientes,
ofertas e eventos transacionais para utilização nas etapas de análise exploratória e modelagem preditiva.